# 07 · Re-Identification y conservación de identidades (≈2 h)

En el notebook 06 vimos que **la mitad de los ID switches eran desapariciones largas**: el Kalman no puede
"recordar" a alguien que sale 2 segundos del encuadre. La solución es la **apariencia**: un modelo que convierte
el recorte de una persona en un vector (*embedding*) tal que la misma persona → vectores cercanos.

Objetivos:
1. Qué es ReID, cómo se entrena (*metric learning*, triplet loss ✍️) y cómo se evalúa (rank-1, mAP).
2. Distancia coseno y galerías ✍️.
3. Usar la apariencia **online** en el tracker (estilo DeepSORT/BoT-SORT).
4. **Stitching offline** de tracklets ✍️ → gran salto de IDF1.
5. Extraer embeddings reales de un vídeo y clasificar equipos por color.

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte, clona el repo y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
if ROOT is None:   # este notebook usa código del repo (vision_utils/ o 11_produccion/) → clonarlo
    ROOT = Path.cwd() / "vision-bootcamp"
    url = "https://github.com/jcandel01/vision-bootcamp.git"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(ROOT)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo: " + r.stderr)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

from vision_utils.synthetic import make_scene
from vision_utils.mot_metrics import evaluate
from vision_utils.tracker import run_tracker
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
pd.set_option("display.width", 200)

## 1. ¿Qué es ReID?

**Person re-identification**: dada una imagen *query* de una persona, encontrar en una **galería** las imágenes
de la misma persona (otra cámara, otro momento). Se plantea como **aprendizaje métrico**: aprender una función
$f(\text{recorte}) \in \mathbb{R}^d$ (d = 128–2048), normalizada a norma 1, tal que
$\cos(f(a), f(b))$ sea alta si son la misma identidad.

- **Entrenamiento**: datasets con IDs (Market-1501, MSMT17; en deporte, SoccerNet-ReID). Pérdidas:
  - *ID loss*: clasificación softmax sobre las identidades de train (luego se tira la cabeza y se usa el embedding).
  - **Triplet loss**: para un *anchor* a, un *positivo* p (misma ID) y un *negativo* n:
    $L = \max(0,\; d(a,p) - d(a,n) + m)$ → el positivo debe estar más cerca que el negativo, con margen m.
    **Batch-hard mining**: en cada batch (P identidades × K imágenes), usar el positivo más lejano y el negativo
    más cercano de cada anchor.
  - Lo habitual: ID loss + triplet (+ trucos de "Bag of Tricks": BNNeck, label smoothing, random erasing).
- **Modelos**: OSNet (ligero, omni-scale; el default en BoxMOT/StrongSORT), ResNet50-BoT, FastReID, CLIP-ReID,
  ViT/TransReID. En la práctica de tracking: modelos pequeños porque se ejecutan **por cada detección y frame**.
- **Evaluación**: **rank-1** (¿el más parecido de la galería es correcto?) y **mAP** sobre el ranking.
  Importante: las identidades de test **no aparecen en train** (si no, es clasificación, no ReID).

**Limitaciones en deporte**: todos los jugadores del mismo equipo llevan la misma ropa → la apariencia
distingue bien **equipos** pero mal **jugadores del mismo equipo**. Lo que sí los distingue: **dorsal** (OCR de
números, p. ej. con PARSeq; reto SoccerNet Jersey Number), complexión, botas, peinado, y la **continuidad
espacio-temporal** (un jugador no se teletransporta).

## 2. ✍️ EJERCICIO: distancia coseno y búsqueda en galería

Con embeddings L2-normalizados, similitud coseno = producto escalar, y $\|a-b\|^2 = 2 - 2\cos(a,b)$.
Implementa `cosine_dist(A, B)` → matriz `1 − cos` de tamaño `(N, M)`, y `rank1(query_emb, query_ids,
gallery_emb, gallery_ids)`.

In [ ]:
# SOLUCIÓN
def cosine_dist(A, B):
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return 1 - A @ B.T

def rank1(qe, qid, ge, gid):
    D = cosine_dist(qe, ge)
    return float(np.mean(gid[D.argmin(1)] == qid))

In [ ]:
# Embeddings simulados de la escena sintética: query = detecciones de la 1ª mitad, galería = 2ª mitad
scene = make_scene(seed=0)
E = np.concatenate(scene.det_emb); I = np.concatenate(scene.det_gt_id)
Fr = np.concatenate([np.full(len(d), f) for f, d in enumerate(scene.dets)])
valid = I > 0
q = valid & (Fr < 200); g = valid & (Fr >= 200)
print(f"rank-1 = {rank1(E[q][::10], I[q][::10], E[g], I[g]):.3f}")
D = cosine_dist(E[valid][::40], E[valid][::40]); ids = I[valid][::40]; o = np.argsort(ids)
plt.imshow(D[o][:, o], cmap="viridis"); plt.title("distancia coseno (ordenado por ID): bloques = misma persona"); plt.colorbar();

## 3. ✍️ EJERCICIO: triplet loss con batch-hard mining (PyTorch)

`emb: (B, d)`, `labels: (B,)`. Para cada anchor: `d_ap` = distancia **máxima** a un positivo, `d_an` =
distancia **mínima** a un negativo; `loss = mean(relu(d_ap − d_an + margin))`.

In [ ]:
# SOLUCIÓN
def batch_hard_triplet(emb, labels, margin=0.3):
    emb = F.normalize(emb, dim=1)
    d = torch.cdist(emb, emb)                                  # (B, B) distancias euclídeas
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(labels), dtype=torch.bool, device=emb.device)
    d_ap = d.masked_fill(~same | eye, -1).max(1).values        # positivo más lejano
    d_an = d.masked_fill(same, float("inf")).min(1).values     # negativo más cercano
    return F.relu(d_ap - d_an + margin).mean()

In [ ]:
# Mini entrenamiento: cada "apariencia" = 16 dims que dependen de la identidad + 48 dims de variación molesta
# (iluminación, fondo, pose) con mucha varianza. La red debe aprender a ignorar lo molesto.
# (Con imágenes reales sería igual: backbone CNN en vez del MLP, y un sampler P×K.)
torch.manual_seed(0)
n_id, P, K = 50, 8, 4
protos = torch.randn(n_id, 16)
def appearance(ids):
    return torch.cat([protos[ids] + 0.5 * torch.randn(len(ids), 16), 2.0 * torch.randn(len(ids), 48)], 1)
def sample_batch():
    ids = torch.randperm(n_id)[:P].repeat_interleave(K)
    x = appearance(ids)
    return x, ids

net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 16))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
hist = []
for step in range(1500):
    x, y = sample_batch()
    loss = batch_hard_triplet(net(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    hist.append(loss.item())

with torch.no_grad():
    yq, yg = torch.arange(n_id), torch.arange(n_id).repeat(3)
    xq, xg = appearance(yq), appearance(yg)
    r_raw = rank1(xq.numpy(), yq.numpy(), xg.numpy(), yg.numpy())
    r_net = rank1(net(xq).numpy(), yq.numpy(), net(xg).numpy(), yg.numpy())
plt.plot(hist); plt.title("triplet loss"); plt.show()
print(f"rank-1 con la entrada cruda: {r_raw:.2f} | con embedding aprendido: {r_net:.2f}")

## 4. Apariencia online en el tracker

Cómo se combina con el movimiento (lo que debes saber explicar):
- **DeepSORT**: coste $c = \lambda\, d_{Mahalanobis} + (1-\lambda)\, d_{coseno}$ (en la práctica λ≈0) con
  puertas: se descartan parejas lejanas según Mahalanobis **o** con distancia coseno > umbral. Cada track guarda
  una **galería** de sus últimos ~100 embeddings y se usa la distancia mínima. **Matching cascade**: primero se
  asocian los tracks vistos más recientemente.
- **BoT-SORT**: $c = \min(c_{IoU}, c_{app})$, pero $c_{app}$ solo vale si la apariencia es muy parecida
  (`appearance_thresh`) **y** las cajas están cerca (`proximity_thresh`) → la apariencia nunca empareja cosas
  lejanas.
- **EMA del embedding** (JDE, StrongSORT, Deep OC-SORT): $e_t = \alpha e_{t-1} + (1-\alpha) f_t$ con α≈0.9 →
  robusto a un frame malo; Deep OC-SORT baja el peso de embeddings de detecciones ocluidas o poco confiables.
- **No actualizar la apariencia** con detecciones ocluidas o de baja confianza (contaminan con el oclusor).

Nuestro `Tracker(use_reid=True)` (ver `vision_utils/tracker.py`) mezcla 1−IoU y distancia coseno, con una
puerta espacial que se ensancha con el tiempo perdido, para poder recuperar tracks tras oclusiones:

In [ ]:
runs = {"ByteTrack": run_tracker(scene, mode="byte"),
        "ByteTrack + ReID": run_tracker(scene, mode="byte", use_reid=True),
        "ByteTrack + ReID, max_age=90": run_tracker(scene, mode="byte", use_reid=True, max_age=90)}
pd.DataFrame([evaluate(scene.gt, v, k) for k, v in runs.items()]).round(3)

**Coste**: un modelo ReID por cada detección y frame. 20 personas a 25 FPS = 500 recortes/s. Mitigaciones:
batch de todos los recortes del frame, modelo pequeño (OSNet-x0.25) en FP16/TensorRT, calcular apariencia solo
cuando hace falta (tracks ambiguos/perdidos), o usar las *features* del propio detector (Ultralytics `model: auto`
en BoT-SORT; JDE/FairMOT).

## 5. ✍️ EJERCICIO: stitching offline de tracklets

Si el análisis es post-partido, podemos mirar **todo el vídeo** y unir trozos de trayectoria (*tracklets*) que
son la misma persona. Algoritmo:
1. Para cada tracklet: frame inicial/final, posición inicial/final (pie de la caja) y **embedding medio**
   (media normalizada de los embeddings de sus detecciones, idealmente solo las de alta confianza).
2. Candidatos (A → B): B empieza **después** de que A termina, con hueco ≤ `max_gap` frames, y la distancia
   recorrida es físicamente posible (`dist ≤ max_speed · gap`).
3. Coste = distancia coseno de los embeddings; descartar si > `app_thr`.
4. Asignación **húngara** (cada final se une a lo sumo con un inicio) y renombrar IDs encadenando las uniones.

Implementa `stitch(mot, emb_of_row, max_gap, max_speed, app_thr)` donde `emb_of_row[k]` es el embedding de la
fila k de `mot`. Devuelve una copia de `mot` con los IDs fusionados.

In [ ]:
# SOLUCIÓN
def stitch(mot, emb_of_row, max_gap=150, max_speed=8.0, app_thr=0.25):
    ids = np.unique(mot[:, 1]).astype(int)
    info = {}
    for tid in ids:
        m = mot[:, 1] == tid
        t = mot[m]; o = np.argsort(t[:, 0]); t = t[o]
        e = emb_of_row[m][o]
        hi = t[:, 6] >= 0.5 if t.shape[1] > 6 else np.ones(len(t), bool)   # solo detecciones fiables
        e = e[hi].mean(0) if hi.any() else e.mean(0)
        foot = lambda r: np.array([(r[2] + r[4]) / 2, r[5]])
        info[tid] = dict(start=t[0, 0], end=t[-1, 0], p0=foot(t[0]), p1=foot(t[-1]), emb=e / np.linalg.norm(e))
    C = np.full((len(ids), len(ids)), 1e5)
    for a, ta in enumerate(ids):
        for b, tb in enumerate(ids):
            A, B = info[ta], info[tb]
            gap = B["start"] - A["end"]
            if a == b or not (0 < gap <= max_gap):
                continue
            if np.linalg.norm(B["p0"] - A["p1"]) > max_speed * gap + 50:
                continue
            d = 1 - A["emb"] @ B["emb"]
            if d < app_thr:
                C[a, b] = d
    r, c = linear_sum_assignment(C)
    nxt = {ids[i]: ids[j] for i, j in zip(r, c) if C[i, j] < 1e5}
    # encadenar: A→B→C pasan todos a ser A
    prev = {v: k for k, v in nxt.items()}
    root = {}
    for tid in ids:
        x = tid
        while x in prev:
            x = prev[x]
        root[tid] = x
    out = mot.copy()
    out[:, 1] = [root[int(t)] for t in mot[:, 1]]
    return out, nxt

In [ ]:
byte = runs["ByteTrack"]
emb_rows = np.array([scene.det_emb[int(f) - 1][int(j)] for f, j in zip(byte[:, 0], byte[:, 7])])
stitched, links = stitch(byte, emb_rows)
print(f"{len(links)} uniones de tracklets: {np.unique(byte[:, 1]).size} IDs → {np.unique(stitched[:, 1]).size} IDs "
      f"(personas reales: {scene.meta['n_people']})")

from vision_utils.postprocess import interpolate_tracks   # la solución del ejercicio del notebook 06

pd.DataFrame([evaluate(scene.gt, byte, "ByteTrack (online)"),
              evaluate(scene.gt, stitched, "+ stitching ReID"),
              evaluate(scene.gt, interpolate_tracks(stitched, 30), "+ stitching + interpolación")]).round(3)

Este es el patrón de un sistema **offline** de calidad: tracker online barato → stitching por apariencia y
física → interpolación/suavizado. Es lo que hacen StrongSORT (AFLink + GSI) y los pipelines de SoccerNet.

Riesgos del stitching: unir dos personas distintas (*false merge*) es peor que dejar dos tracklets separados
(un merge erróneo contamina toda la trayectoria). Por eso umbrales conservadores y restricciones físicas
(y en deporte: mismo equipo, mismo dorsal si se ha leído).

## 6. Embeddings reales y clasificación de equipos

Dos *features* de apariencia baratas que puedes calcular sobre los recortes de un vídeo real:
1. **Embedding de una CNN preentrenada** (ResNet18 ImageNet, sin cabeza). No es un modelo ReID entrenado, pero
   sirve de baseline y para ver el flujo. Para producción: OSNet/BoT entrenado en ReID (BoxMOT trae pesos).
2. **Histograma de color HSV del torso** → con k-means (k=2) separa **equipos** (y un 3er cluster para
   árbitro/porteros). Truco: recortar la parte superior central de la caja (torso) para evitar césped y piernas.

Aquí: detectamos personas en un frame del vídeo, sacamos ambos descriptores y agrupamos.

In [ ]:
import cv2
from ultralytics import YOLO
from torchvision.models import resnet18, ResNet18_Weights

cap = cv2.VideoCapture(str(DATA / "people.mp4")); ok, frame = cap.read(); cap.release()
det = YOLO(str(DATA / "yolo11n.pt"))(frame, classes=[0], conf=0.4, device=DEVICE, verbose=False)[0]
boxes = det.boxes.xyxy.cpu().numpy().astype(int)

def torso_hist(img, b, bins=(16, 4)):
    x1, y1, x2, y2 = b
    h, w = y2 - y1, x2 - x1
    crop = img[y1 + int(0.15 * h): y1 + int(0.5 * h), x1 + int(0.2 * w): x2 - int(0.2 * w)]
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, bins, [0, 180, 0, 256]).ravel()
    return hist / (hist.sum() + 1e-9)

H = np.array([torso_hist(frame, b) for b in boxes])
crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-3)
_, team, _ = cv2.kmeans(H.astype(np.float32), 2, None, crit, 5, cv2.KMEANS_PP_CENTERS)

backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1); backbone.fc = nn.Identity()
backbone = backbone.eval().to(DEVICE)
tf = ResNet18_Weights.IMAGENET1K_V1.transforms()
crops = [torch.from_numpy(cv2.resize(frame[y1:y2, x1:x2], (128, 256))[:, :, ::-1].copy()).permute(2, 0, 1)
         for x1, y1, x2, y2 in boxes]
with torch.inference_mode():
    emb = backbone(torch.stack([tf(c) for c in crops]).to(DEVICE)).cpu().numpy()
print("embeddings:", emb.shape)

vis = frame.copy()
for b, t in zip(boxes, team.ravel()):
    cv2.rectangle(vis, tuple(b[:2]), tuple(b[2:]), (0, 0, 255) if t else (255, 0, 0), 3)
plt.figure(figsize=(14, 8)); plt.imshow(vis[:, :, ::-1]); plt.axis("off")
plt.title("k-means (k=2) sobre histograma HSV del torso (en un partido = equipos)");